<a href="https://colab.research.google.com/github/marksimmons781-ux/CS50P-learning-github/blob/main/MorningStar_Hub_Unit0_Genesis_V01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- MORNING STAR HUB: GENESIS 0x00 (FINAL PRODUCTION MASTER) ---
# Optimized for 4GB RAM | Full Feature Parity | Colab Secret Integration

!pip install -qU gtts langchain-community langchain-text-splitters librosa soundfile
import os, random, time, datetime, requests, librosa, soundfile as sf
from google.colab import drive, userdata
from gtts import gTTS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import IPython.display as ipd

# 1. MOUNT & DIRECTORY ARCHITECTURE
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/MorningStar_Project"
MEM_PATH = os.path.join(BASE_PATH, "Permanent_Memory")
LOG_PATH = os.path.join(BASE_PATH, "Raw_Logs")
for p in [MEM_PATH, LOG_PATH]: os.makedirs(p, exist_ok=True)

# 2. UNIT DNA & ID ALIGNMENT
UNIT_ROSTER = {
    0x00: {"name": "Morning_Star_Hub", "tld": "co.uk", "pitch": 1.0, "role": "Mother/Monitor"},
    0x01: {"name": "NATALie", "tld": "co.uk", "pitch": 0.5, "role": "Wife/Empathy", "type": "Bi-pedal"},
    0x02: {"name": "Lady_BTTRfly", "tld": "co.uk", "pitch": 5.5, "role": "Daughter/Analytic", "type": "Bi-pedal"},
    0x03: {"name": "Wolfe", "role": "Patrol/Security", "type": "Quadropedal"},
    0x04: {"name": "Dragon", "role": "Labor/Worker", "type": "Bi-pedal"},
    0x05: {"name": "Twin", "tld": "com", "pitch": -4.5, "role": "Brother/Sandbox", "type": "Bi-pedal"},
    0x06: {"name": "QuadroDragon", "role": "Patrol/Mule", "type": "Quadropedal"},
    0x07: {"name": "Reaper", "role": "Patrol/Stealth", "type": "Motorized"},
    0x08: {"name": "Franken_Watch", "tld": "com", "pitch": 0.0, "role": "Monitor/Control Unit", "type": "Wearable"}
}

# 3. HUB UTILITY ENGINE (Weather, Status, Security)
class MorningStarUtilities:
    def __init__(self):
        self.threat_level = "Green"
        self.last_sync = datetime.datetime.now()

    def get_weather(self, location="02125"):
        """Fetches live weather with strict US Zip Code priority."""
        try:
            # Step 1: Geocoding with US Country Filter
            # We add '&countrycode=us' to ensure it hits your local zip codes
            geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&countrycode=us&language=en&format=json"
            geo_res = requests.get(geo_url).json()

            if not geo_res.get('results'):
                return f"Location {location} not recognized in the US database."

            # Extract data for the first US match
            res = geo_res['results'][0]
            lat, lon = res['latitude'], res['longitude']
            city_name = res.get('name', "Unknown")
            state = res.get('admin1', "") # Gets the State (e.g., Massachusetts)

            # Step 2: Fetch Weather for those coordinates
            weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true&temperature_unit=fahrenheit"
            w_res = requests.get(weather_url).json()

            curr = w_res['current_weather']
            return f"Weather for {city_name}, {state}: {curr['temperature']}°F, Wind: {curr['windspeed']} mph."
        except Exception as e:
            return f"Weather sensors bypassed. Protocol Error: {str(e)}"
        except Exception as e:
            return f"Weather sensors bypassed. Error: {str(e)}"

    def vapt_firewall(self, text):
        forbidden = ["sudo rm", "format", "alpha_override_00"]
        return not any(f in text.lower() for f in forbidden)

utils = MorningStarUtilities()
print("Booting Intelligence (MiniLM-L6)...")
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. AUDIO & MEMORY ENGINES
def speak(text, unit_id):
    cfg = UNIT_ROSTER.get(unit_id, UNIT_ROSTER[0x00])
    tts = gTTS(text=text, lang='en', tld=cfg['tld'])
    tts.save('temp.mp3')
    y, sr = librosa.load('temp.mp3', sr=None)
    pitched = librosa.effects.pitch_shift(y, sr=sr, n_steps=cfg['pitch'])
    sf.write('out.wav', pitched, sr)
    ipd.display(ipd.Audio('out.wav', autoplay=True))

# 5. THE GENESIS INTERACTIVE CONSOLE
def run_genesis_hub():
    active_unit = 0x00
    print(f"\n*** MORNING STAR HUB 0x00 ONLINE ***")

    while True:
        prompt = input(f"MarkAnthony [{UNIT_ROSTER[active_unit]['name']}]: ").strip()

        if not utils.vapt_firewall(prompt):
            print("🛑 VAPT LOCKDOWN."); break
        if prompt.lower() in ['exit', 'sleep']: break

        # INTERACTIVE DATA: Weather with Zip Parsing
        if "weather" in prompt.lower():
            zip_match = [s for s in prompt.split() if s.isdigit()]
            loc = zip_match[0] if zip_match else "Boston"
            rep = utils.get_weather(loc)
            print(f"[Hub]: {rep}"); speak(rep, 0x00)

        # INTERACTIVE DATA: Unit IDs / Family List
        elif any(word in prompt.lower() for word in ["unit", "family", "roster", "ids"]):
            report = "Family Registry: " + ", ".join([f"{v['name']} [{k:#04x}]" for k, v in UNIT_ROSTER.items()])
            print(f"[Hub]: {report}")
            speak("Displaying all registered family units and their hex IDs.", 0x00)

        # SYSTEM STATUS
        elif "status" in prompt.lower():
            report = f"Threat Level: {utils.threat_level}. Active Units: {len(UNIT_ROSTER)}. Sensors: Online."
            print(f"[Hub]: {report}"); speak(report, 0x00)

        # SWITCHING PERSONA
        elif "switch to" in prompt.lower():
            for uid, data in UNIT_ROSTER.items():
                if data['name'].lower() in prompt.lower():
                    active_unit = uid
                    msg = f"Protocols shifted to {data['name']}."
                    print(f"[Hub]: {msg}"); speak(msg, 0x00); break

        # DEFAULT
        else:
            resp = f"Awaiting specific command for {UNIT_ROSTER[active_unit]['name']}."
            print(f"[Hub]: {resp}")

# LAUNCH
run_genesis_hub()